## **Modelación del Modelo y Selección de Variables**

* En este capítulo se aborda el proceso de modelación del precio de los vehículos, así como la selección de las variables más relevantes para el desempeño del modelo. Si bien en etapas previas del análisis exploratorio se identificó que variables como el año del vehículo y el kilometraje presentan una fuerte influencia sobre el precio, en esta fase se evalúa su impacto dentro del modelo predictivo, junto con otras variables categóricas y numéricas.

* Es importante destacar que, en el mercado automotriz, los vehículos nuevos experimentan una depreciación significativa, cercana al 40%, incluso cuando han recorrido pocos kilómetros fuera de la agencia. Este comportamiento justifica la inclusión del año y del kilometraje como variables clave, ya que capturan tanto el efecto del paso del tiempo como el desgaste del vehículo.

* Durante el proceso de modelación, se analizó el comportamiento del modelo ante distintas combinaciones de variables, evaluando su capacidad de generalización mediante métricas como el coeficiente de determinación ($R^2$) y validación cruzada. Asimismo, se examinó la presencia de sobreajuste (overfitting) y subajuste (underfitting), comparando el desempeño del modelo en los conjuntos de entrenamiento, validación y prueba.

* A partir de este análisis, se tomaron decisiones informadas sobre la inclusión o exclusión de variables, buscando un equilibrio entre la complejidad del modelo y su capacidad de generalización, con el objetivo de construir un modelo robusto y consistente con la dinámica real del mercado automotriz.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/dataset/car_sales_data_clean.csv')

In [ ]:
df.head()

,Fabricante,Modelo,Motor,Combustible,Año,Kilometraje,Precio
0,Ford,Fiesta,1.0,Petrol,2002,127300,3074
1,Porsche,718 Cayman,4.0,Petrol,2016,57850,49704
2,Ford,Mondeo,1.6,Diesel,2014,39190,24072
3,VW,Polo,1.0,Petrol,2006,127869,4101
4,Ford,Focus,1.4,Petrol,2018,33603,29204


##

In [ ]:
X = df.drop(['Precio'], axis=1)
y = df['Precio']

## **Partición del Conjunto de Datos**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train,X_val,y_train,y_val = train_test_split(X_train,y_train,test_size=0.25,random_state=42)

* El conjunto de datos se dividió en tres subconjuntos: **entrenamiento**, **validación** y **prueba**. Inicialmente, se reservó el 20% de los datos para el conjunto de prueba, el cual se utilizó exclusivamente para evaluar el desempeño final del modelo. Posteriormente, el 80% restante se dividió en un 75% para entrenamiento y un 25% para validación, resultando en una partición final de 60% entrenamiento, 20% validación y 20% prueba.

## **Primer modelo**

In [ ]:
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder,PowerTransformer

In [ ]:
num_features = [ 'Motor','Año','Kilometraje']
cat_features = ['Fabricante', 'Modelo','Combustible']


preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ]
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

In [ ]:
model1 = TransformedTargetRegressor(
    regressor=Pipeline(steps=[
        ('preprocess', preprocessor),
        ('model', LinearRegression())
    ]),
    func=np.log,
    inverse_func=np.exp
)

In [ ]:
model1.fit(X_train, y_train)

TransformedTargetRegressor(func=<ufunc 'log'>, inverse_func=<ufunc 'exp'>,
                           regressor=Pipeline(steps=[('preprocess',
                                                      ColumnTransformer(transformers=[('num',
                                                                                       StandardScaler(),
                                                                                       ['Motor',
                                                                                        'Año',
                                                                                        'Kilometraje']),
                                                                                      ('cat',
                                                                                       OneHotEncoder(handle_unknown='ignore'),
                                                                                       ['Fabricante',
                                                                                        'Modelo',
                                                                                        'Combustible'])])),
                                                     ('model',
                                                      LinearRegression())]))

La función **TransformedTargetRegressor** permite aplicar una transformación matemática directamente sobre la variable objetivo (target) sin necesidad de modificar manualmente los datos originales.


* La función func = np.log transforma el precio a escala logarítmica durante el entrenamiento.

* El modelo (en este caso una Regresión Lineal) aprende sobre esta escala transformada, lo que:

* Reduce la asimetría del target

* Disminuye el impacto de valores extremos

* Linealiza relaciones no lineales (como la depreciación de los autos)

Posteriormente:

La función inverse_func = np.exp revierte automáticamente la predicción a la escala original del precio (dólares).

Esto permite:

* Entrenar el modelo en una escala más estable

* Evaluar e interpretar resultados en unidades reales

* En otras palabras, el flujo es:

* Precio original → log(precio)

* Entrenamiento del modelo en la escala logarítmica

* Predicción → exp(predicción) → precio real

## **Métricas**

### **Coeficiente de determinación**

In [ ]:
model1.score(X_train,y_train)

0.9912428113757339

In [ ]:
model1.score(X_test,y_test)

0.9912708321898883

In [ ]:
model1.score(X_val,y_val)

0.991227501955559

* En el primer modelo, al mantener la variable Modelo, la cual es de tipo categórica nominal (es decir, no existe un orden inherente entre sus categorías), se obtuvo un **R²**  superior a 0.99, lo que indica un alto poder explicativo del modelo.

* No obstante, este desempeño tan elevado puede ser indicio de un posible **sobreajuste** (overfitting), ya que el modelo podría estar aprendiendo patrones demasiado específicos del conjunto de entrenamiento. En este escenario, el modelo puede depender de combinaciones particulares de variables que cumplen ciertas condiciones específicas, reduciendo su capacidad de generalizar correctamente a datos no vistos.

* Aunque resulta atractivo contar con un modelo que reporte un **R²** cercano a 0.99, en la práctica esto no siempre refleja un comportamiento realista del mercado. En muchos casos, modelos con métricas extremadamente altas no se traducen en un buen desempeño en producción, especialmente cuando los datos futuros presentan variaciones distintas a las observadas durante el entrenamiento.

### **Error absoluto medio**

In [ ]:
y_train_pred = model1.predict(X_train)
y_test_pred = model1.predict(X_test)
y_val_pred = model1.predict(X_val)

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score

In [ ]:
print('MAE train: ',mean_absolute_error(y_train,y_train_pred))
print('MAE test: ',mean_absolute_error(y_test,y_test_pred))
print('MAE val: ',mean_absolute_error(y_val,y_val_pred))

MAE train:  663.6717994070796
MAE test:  644.793674661757
MAE val:  674.9826096895467


| Modelo                        | MAE Train | MAE Test | MAE Val |
|------------------------------|-----------|----------|---------|
| Datos sin tratar             | 5,779.99  | 5,781.50 | 5,678.01 |
| Datos tratados               | 663.67    | 644.79   | 674.98  |


* El modelo entrenado con datos previamente limpiados y transformados presenta una mejora sustancial en el desempeño predictivo respecto al modelo entrenado con datos sin tratamiento. En particular, el error absoluto medio (MAE) se reduce de valores cercanos a las 5,800 unidades a aproximadamente 650 USD en los conjuntos de entrenamiento, prueba y validación.

* Esta diferencia evidencia que la calidad de los datos tiene un impacto determinante en la capacidad predictiva del modelo, incluso manteniendo el mismo algoritmo de regresión. No obstante, es importante destacar que este desempeño corresponde específicamente al conjunto de datos analizado. La inclusión de variables altamente informativas del precio, así como la estructura del dataset, puede limitar la capacidad de generalización del modelo frente a vehículos más recientes o configuraciones no presentes en el conjunto original.

### **Validación cruzada**

In [ ]:
cross_val_score(model1,X_train,y_train,cv=5)

array([0.99168808, 0.9908805 , 0.99167021, 0.98983844, 0.99198717])

In [ ]:
cross_val_score(model1,X_test,y_test,cv=5)

array([0.98963534, 0.99298921, 0.99080213, 0.99217911, 0.99196208])

In [ ]:
cross_val_score(model1,X_val,y_val,cv=5)

array([0.99399166, 0.99044398, 0.99011491, 0.99107334, 0.98849112])

## **Segundo modelo**

In [ ]:
num_features = [ 'Motor','Año','Kilometraje']
cat_features = ['Fabricante', 'Combustible']


preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ]
)

In [ ]:
model2 = TransformedTargetRegressor(
    regressor=Pipeline(steps=[
        ('preprocess', preprocessor),
        ('model', LinearRegression())
    ]),
    func=np.log,
    inverse_func=np.exp
)

In [ ]:
model2.fit(X_train, y_train)

TransformedTargetRegressor(func=<ufunc 'log'>, inverse_func=<ufunc 'exp'>,
                           regressor=Pipeline(steps=[('preprocess',
                                                      ColumnTransformer(transformers=[('num',
                                                                                       StandardScaler(),
                                                                                       ['Motor',
                                                                                        'Año',
                                                                                        'Kilometraje']),
                                                                                      ('cat',
                                                                                       OneHotEncoder(handle_unknown='ignore'),
                                                                                       ['Fabricante',
                                                                                        'Combustible'])])),
                                                     ('model',
                                                      LinearRegression())]))

## **Métricas**

### **Coeficiente de determinación**

In [ ]:
model2.score(X_train,y_train)

0.9314139104522428

In [ ]:
model2.score(X_test,y_test)

0.9244809587016942

In [ ]:
model2.score(X_val,y_val)

0.9358183794308487

* Al excluir la variable Modelo, la cual representa una categoría nominal altamente granular y potencialmente correlacionada de forma directa con el precio, el desempeño del modelo se estabiliza en un $R²$ cercano a 0.92 tanto en validación cruzada como en los conjuntos de entrenamiento, prueba y validación.

### **Validación cruzada**

In [ ]:
cross_val_score(model2,X_train,y_train,cv=5)

array([0.92664443, 0.93798856, 0.94006324, 0.92796284, 0.92507216])

In [ ]:
cross_val_score(model2,X_test,y_test,cv=5)

array([0.92790025, 0.92731193, 0.91080329, 0.94856039, 0.92201638])

In [ ]:
cross_val_score(model2,X_val,y_val,cv=5)

array([0.93679587, 0.94340836, 0.93145378, 0.91850909, 0.94145752])

### **Error absoluto medio**

In [ ]:
y_train_pred = model2.predict(X_train)
y_test_pred = model2.predict(X_test)
y_val_pred = model2.predict(X_val)

In [ ]:
print('MAE train: ',mean_absolute_error(y_train,y_train_pred))
print('MAE test: ',mean_absolute_error(y_test,y_test_pred))
print('MAE val: ',mean_absolute_error(y_val,y_val_pred))

MAE train:  1870.5118405895928
MAE test:  1866.934336929654
MAE val:  1844.6791419531476




* Aunque el **error absoluto medio** (MAE) aumenta respecto al modelo inicial, la consistencia de esta métrica entre las distintas particiones indica una mejora en la capacidad de generalización del modelo y una reducción del riesgo de sobreajuste.

| Modelo                                   | MAE Train (USD) | MAE Test (USD) | MAE Val (USD) |
|------------------------------------------|-----------------|----------------|---------------|
| Modelo 1: Datos sin limpiar              | 5,779.99        | 5,781.50       | 5,678.01     |
| Modelo 2: Datos limpiados                | 663.67          | 644.79         | 674.98       |
| Modelo 3: Datos limpiados (sin variable "Modelo")   | 1,870.51        | 1,866.93       | 1,844.68     |



- El Modelo 2 presenta el menor error absoluto medio (MAE), evidenciando el impacto positivo de la limpieza y transformación de los datos.  
- El Modelo 3, al excluir la variable *Modelo*, sacrifica precisión a favor de una mayor capacidad de generalización y menor riesgo de fuga de información.  
- El Modelo 1 refleja el desempeño esperado al entrenar un modelo sobre datos sin tratamiento, con errores significativamente más altos.


In [ ]:
def predi

In [ ]:
X.columns

Index(['Fabricante', 'Modelo', 'Motor', 'Combustible', 'Año', 'Kilometraje'], dtype='object')

In [ ]:
def Predict(Fabricante,Motor,Combustible,Año,Kilometraje):
  x = pd.DataFrame(
      {
          'Fabricante':[Fabricante],
          'Motor':[Motor],
          'Combustible':[Combustible],
          'Año':[Año],
          'Kilometraje':[Kilometraje]
      }
  )

  return model2.predict(x)

In [ ]:
Predict('Toyota',1.8,'Petrol',2016,150000)

array([10922.4289896])

In [ ]:
import joblib

In [ ]:
joblib.dump(model2,'/content/drive/MyDrive/modelo/model2.pkl')

['/content/drive/MyDrive/modelo/model2.pkl']

## **Conclusión**

* Dado lo anterior, la elección natural es el Modelo 2, ya que presenta el menor error absoluto medio, con un margen de error cercano a los 650 dólares, además de un comportamiento estable entre los conjuntos de entrenamiento, prueba y validación. Esto indica que el modelo logra capturar de forma adecuada la relación entre las variables explicativas y el precio del vehículo dentro del conjunto de datos analizado.

* No obstante, es importante señalar que este desempeño está condicionado a las características del dataset utilizado. La inclusión de variables altamente informativas del precio puede limitar la capacidad de generalización del modelo cuando se enfrenta a vehículos más recientes, configuraciones no presentes en los datos originales o cambios en las condiciones del mercado.

* Por esta razón, aunque el Modelo 2 es el más preciso en el contexto actual, el Modelo 3 representa una alternativa más conservadora y robusta desde el punto de vista de generalización, al reducir el riesgo de fuga de información a costa de un mayor margen de error.